# E4 · Adaptación LoRA de CLIP sobre E3-100

**Objetivo.** Medir el beneficio adicional de adaptar el encoder visual después de E3.

**Pregunta.** ¿LoRA en la atención de CLIP mejora la recuperación frente a E3-100 sin perder transferencia a PetTrace?

**Hipótesis.** Una actualización de bajo rango puede representar señales de identidad que la MLP sobre embeddings congelados no captura.

| Elemento | Diseño fijado |
|---|---|
| Intervención | LoRA `r=8`, `alpha=16` en `q_proj` y `v_proj` de CLIP ViT-B/32 |
| Congelado | YOLO, pesos base/proyección de CLIP y cabeza E3-100 |
| Control | E3-100 semilla 46, seleccionada entre 45–49 solo por validación |
| Muestra | Fuentes públicas refinadas `v2`, equivalentes a `pettrace-training@v4.100`; PetTrace no participa del entrenamiento |
| Selección | mAP macro por especie en validación pública; nunca por test |
| Evaluación | Test público v1 y PetTrace v4, con galerías separadas |
| Estado | **E4 v2, semilla 46, publicado en ambos dominios; candidato 19 sin promoción de alias** |

> Las salidas conservadas abajo corresponden a esta corrida E4 v2 y sus recibos de MLflow.


### Entorno

En una T4 nueva, esta única instalación obtiene el código exacto desde PyPI. Las credenciales se leen de Colab Secrets y nunca se imprimen.

In [1]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks]==0.2.27"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import json, os, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata
import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if not (3, 11) <= sys.version_info[:2] < (3, 14) or sys.platform != "linux":
    raise RuntimeError("E4 requires Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.27":
    raise RuntimeError("Install PetTrace 0.2.27 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")
EXPERIMENT, DATASET = "E4", "v4"
DATASET_COMMIT = "8b12db0dce41040a821f9792bea5b352342263ce"
TRAINING_DATASET_VERSION, SOURCE_SAMPLE_PCT, SEED = "v2", 100, 46
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
BASE_MODEL = "models:/pettrace-embedder/2"
LOG_TO_MLFLOW = True
REGISTER_E4 = True
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path("/content/reports") / "e4_v4" / RUN_LABEL
REPORT.mkdir(parents=True)
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0), "| Report:", REPORT)

Access [1/2] Loading Colab Secrets…
Access [2/2] Credentials available.
Package: 0.2.27 | GPU: Tesla T4 | Report: /content/reports/e4_v4/20261106T184440Z-b6c3e98d


## Fase 1. Preparación y preprocesamiento de datos

E4 **carga** las fuentes refinadas públicas `v2` desde commits inmutables de Hugging Face; no adquiere ZIP, no limpia y no ejecuta YOLO. Sus imágenes, identidades y splits son exactamente los de `v4.100`; `v2` añade únicamente las cajas verificadas que reproducen el recorte E2.

In [2]:
from pettrace.data import training
from pettrace.data.volume import training_volume

print("Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…")
prepared = training(version=TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/pettrace-refined", token=os.environ["HF_TOKEN"])
manifest_rows = prepared.rows
training_source = prepared.provenance
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
(REPORT / "transport-equivalence.json").write_text(json.dumps(training_source, indent=2))
print("Dataset [2/2] Verified:", prepared.fingerprint, "| photos:", len(manifest_rows),
      "| identities:", prepared.summary["retained_identities"],
      "| commits:", [source.commit[:12] for source in prepared.sources])

Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…
Dataset: verifying and materializing alessandro54/labeled-cats-in-the-wild-refined@581e44a04fd4…
Dataset: lcw ready (131,975 photos).
Dataset: verifying and materializing alessandro54/dogs-of-the-world-refined@9fdd6e5f7eaa…
Dataset: dogs-world ready (61,851 photos).
Dataset [2/2] Verified: d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58 | photos: 193826 | identities: 42113 | commits: ['581e44a04fd4', '9fdd6e5f7eaa']


In [3]:
from pettrace.tracking import production_context
from pettrace.experiments.e3 import load_benchmarks
production_context()
benchmarks = load_benchmarks(manifest_rows, REPORT, private_version=DATASET, private_commit=DATASET_COMMIT)
ds, public_ds, PUBLIC_REPORT = benchmarks.private, benchmarks.public, benchmarks.public_report
print("Public test:", len(public_ds.photos), "photos | PetTrace:", len(ds.photos), "photos")

Benchmarks [1/5] Verifying immutable versions…
Benchmarks [2/5] Loading the private PetTrace benchmark…
Benchmarks [3/5] Verifying 305 private photos…
Benchmarks [4/5] Loading the public v1 test…
Benchmarks [5/5] Checking benchmark separation and leakage…
Benchmarks ready in 286.5s: 277 public · 305 PetTrace.
Public test: 277 photos | PetTrace: 305 photos


## Fase 2. Representación y entrenamiento

Se descarga la cabeza E3-100 del run fijado y se verifica su SHA-256. CLIP usa la revisión inmutable indicada arriba. Solo los adaptadores LoRA son entrenables; antes de ajustar se comprueba que su inicialización reproduce E3.

LoRA sigue a [Hu et al. (2022)](https://arxiv.org/abs/2106.09685). La pérdida batch-hard sigue a [Hermans et al. (2017)](https://arxiv.org/abs/1703.07737). Capas, rango, presupuesto y control son decisiones de esta investigación.

In [4]:
from pettrace.evaluation.core import load_model
from pettrace.experiments.e4 import load_control_head
from pettrace.training.lora import Config, PreparedPreprocessor, build_encoder, control_equivalence

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The fixed E2 base failed verification")

head, control_receipt = load_control_head(REPORT)
preprocessor = PreparedPreprocessor(base_model.pipeline, prepared)

training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
train_rows = [row for row in training_rows if row["split"] == "train"]
gate = [
    next(row for row in sorted(train_rows, key=lambda item: item["sha256"])
         if row["species"] == species)
    for species in ("cat", "dog")
]

checked = preprocessor.validate(gate)
pixels = preprocessor.pixels(gate)
e2 = base_model.pipeline.session.run(
    [base_model.pipeline.output_name],
    {base_model.pipeline.input_name: pixels},
)[0]

config = Config(seed=SEED)
encoder, adapted_layers = build_encoder(head, CLIP_REVISION, config=config)
encoder = encoder.to("cuda")  # Move the newly injected LoRA parameters to the GPU.

parity = control_equivalence(encoder, pixels, e2)
(REPORT / "control-equivalence.json").write_text(json.dumps(
    {
        **parity,
        "cached_crop_photos": checked,
        "adapted_layers": adapted_layers,
    },
    indent=2,
))

print("Control parity:", parity, "| LoRA layers:", len(adapted_layers))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Control parity: {'minimum_cosine': 1.0000001192092896, 'maximum_absolute_error': 2.0116567611694336e-07, 'photos': 2} | LoRA layers: 24


In [5]:
from pettrace.training.lora import fit, trainable_parameters

run_config = {"experiment": EXPERIMENT, "dataset": TRAINING_DATASET_VERSION,
              "dataset_fingerprint": prepared.fingerprint, "clip_revision": CLIP_REVISION,
              "control": control_receipt, "training": asdict(config),
              "trainable_parameters": trainable_parameters(encoder),
              "log_to_mlflow": LOG_TO_MLFLOW, "register_model": REGISTER_E4}
(REPORT / "training-config.json").write_text(json.dumps(run_config, indent=2))
def checkpoint(record):
    print(f"Epoch {record['epoch']:02d} | loss {record['train.loss']:.4f} | "
          f"validation macro mAP {record['val.macro_species_mAP']:.4f}", flush=True)
encoder, training_summary = fit(encoder, training_rows, preprocessor, REPORT,
                                config=config, device="cuda", callback=checkpoint)
(REPORT / "training-summary.json").write_text(json.dumps(training_summary, indent=2))
print("Selected epoch:", training_summary["best_epoch"],
      "| validation macro mAP:", training_summary["best_val_mAP"])

Mounted at /content/drive
Epoch 01 | loss 0.1359 | validation macro mAP 0.7722
{'epoch': 1, 'train.loss': 0.13586836585775017, 'train.active_triplets': 0.67625, 'val.macro_species_mAP': 0.7721946868092509, 'training.optimizer_steps': 100, 'training.examples_seen': 3200, 'training.unique_photos_seen': 3185}
Epoch 02 | loss 0.1350 | validation macro mAP 0.7981
{'epoch': 2, 'train.loss': 0.1349511540075764, 'train.active_triplets': 0.65375, 'val.macro_species_mAP': 0.7981095195675303, 'training.optimizer_steps': 200, 'training.examples_seen': 6400, 'training.unique_photos_seen': 6293}
Epoch 03 | loss 0.1221 | validation macro mAP 0.7732
{'epoch': 3, 'train.loss': 0.12207756397314369, 'train.active_triplets': 0.6371875, 'val.macro_species_mAP': 0.7732126432189989, 'training.optimizer_steps': 300, 'training.examples_seen': 9600, 'training.unique_photos_seen': 9311}
Epoch 04 | loss 0.1208 | validation macro mAP 0.7821
{'epoch': 4, 'train.loss': 0.12083989274222404, 'train.active_triplets': 0

In [6]:
from pettrace.training.lora import export_encoder
model_dir = REPORT / "model"
onnx_hashes = export_encoder(base_model.dir, encoder, model_dir,
    model_version="clip-vit-b32-crop-lora-e3-v4.100@v2", verification_pixels=pixels)
e4_model = load_model(str(model_dir))
if not e4_model.golden_ok:
    raise RuntimeError("Exported E4 model failed golden verification")
print("Exported:", onnx_hashes)

/usr/local/lib/python3.13/dist-packages/transformers/models/clip/modeling_clip.py:243: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not interpolate_pos_encoding and (height != self.image_size or width != self.image_size):


Exported: {'detector.onnx': 'da8f9b9ebea5157cdcefa7bea9fd782249ebdf119f917c07d64e5b41942b9c9d', 'encoder.onnx': '0d6bbd7b5156d3630e8b4a2d2369ea110d78f5a94eaf9481f1bbd23b1d6b66cb'}


## Fase 3. Evaluación y análisis

El checkpoint ya quedó fijado por validación. Ahora se evalúan, por separado, el test público v1 y PetTrace v4. Rank-k usa el máximo score por identidad y mAP ordena fotografías en una galería cerrada de la misma especie. La EER es descriptiva porque se calcula sobre los mismos pares evaluados.

**Limitaciones.** Los benchmarks ya fueron observados en experimentos anteriores; no garantizan separación entre sesiones, búsqueda abierta ni generalización poblacional. Esta corrida exploratoria usa una sola semilla y no estima variabilidad de optimización.

In [7]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import metrics_table, save
from pettrace.evaluation.research import save_benchmark_comparison
from pettrace.experiments.e4 import frozen_control
from pettrace.tracking import save_provenance

public_ev = evaluate(public_ds, e4_model.pipeline, zero_shot=True)
private_ev = evaluate(ds, e4_model.pipeline, zero_shot=True)
public_summary = save(public_ev, e4_model, str(PUBLIC_REPORT))
private_summary = save(private_ev, e4_model, str(REPORT))
save_provenance(PUBLIC_REPORT, e4_model); save_provenance(REPORT, e4_model)
public_e3, public_e3_summary = frozen_control(public_ev, "public", PUBLIC_REPORT)
private_e3, private_e3_summary = frozen_control(private_ev, "private", REPORT)
display(Markdown("### Test público\n" + metrics_table({"E3-100": public_e3_summary, "E4": public_summary})))
display(Markdown("### PetTrace v4\n" + metrics_table({"E3-100": private_e3_summary, "E4": private_summary})))
domain_table, _ = save_benchmark_comparison(public_ev, private_ev, REPORT, EXPERIMENT)
display(Markdown("### Dominios separados\n" + domain_table))
comparison = {domain: {metric: current["global"][metric] - control["global"][metric]
               for metric in ("rank1", "rank2", "mAP")}
              for domain, current, control in (("public", public_summary, public_e3_summary),
                                                ("private", private_summary, private_e3_summary))}
(REPORT / "comparison.json").write_text(json.dumps(comparison, indent=2))
print("E4 − E3:", comparison)

### Test público
| Metric | E3-100 | E4 |
|---|---|---|
| Queries / gallery per species | 277 / 34 animals | 277 / 34 animals |
| Global Rank-1 (chance 2.9 %) | **80.1 %** | **85.6 %** |
| Rank-1 cats / dogs | 74.7 % / 87.8 % | 82.7 % / 89.6 % |
| Global Rank-5 (chance 14.7 %) | 98.6 % | 99.3 % |
| Global mAP | 0.764 | 0.813 |
| mAP cats / dogs | 0.694 / 0.864 | 0.765 / 0.880 |
| EER / threshold | 9.5 % / 0.817 | 8.4 % / 0.804 |
| Minimum similarity, same animal | 0.679 | 0.607 |
| Maximum similarity, different animal | 0.978 | 0.976 |

### PetTrace v4
| Metric | E3-100 | E4 |
|---|---|---|
| Queries / gallery per species | 305 / 16 animals | 305 / 16 animals |
| Global Rank-1 (chance 6.2 %) | **78.4 %** | **83.6 %** |
| Rank-1 cats / dogs | 80.1 % / 76.5 % | 84.6 % / 82.6 % |
| Global Rank-5 (chance 31.2 %) | 99.3 % | 99.7 % |
| Global mAP | 0.688 | 0.722 |
| mAP cats / dogs | 0.711 / 0.664 | 0.752 / 0.690 |
| EER / threshold | 12.8 % / 0.803 | 11.7 % / 0.799 |
| Minimum similarity, same animal | 0.504 | 0.467 |
| Maximum similarity, different animal | 0.981 | 0.972 |

### Dominios separados
| Dataset | Group | Queries | Gallery (identities) | Rank-1 | Rank-2 | mAP |
|---|---|---:|---:|---:|---:|---:|
| Public v1 · test | global | 277 | 34 | 85.56% | 94.95% | 0.8126 |
| Public v1 · test | cat | 162 | 34 | 82.72% | 93.21% | 0.7650 |
| Public v1 · test | dog | 115 | 26 | 89.57% | 97.39% | 0.8796 |
| PetTrace v4 | global | 305 | 16 | 83.61% | 92.79% | 0.7218 |
| PetTrace v4 | cat | 156 | 15 | 84.62% | 95.51% | 0.7522 |
| PetTrace v4 | dog | 149 | 16 | 82.55% | 89.93% | 0.6901 |

E4 − E3: {'public': {'rank1': 0.054151624548736454, 'rank2': 0.014440433212996373, 'mAP': 0.04827705203308763}, 'private': {'rank1': 0.05245901639344264, 'rank2': 0.0, 'mAP': 0.03375200640531206}}


### Publicación obligatoria

La corrida solo termina cuando ambos dominios quedan publicados. Mientras el runtime siga activo, un error de publicación conserva el reporte local y permite reintentar esta celda sin volver a entrenar. MLflow conserva la evidencia publicada y el modelo candidato; el registro no mueve aliases.

In [8]:
from pettrace.experiments.e3 import publish_results, register_candidate
publication = publish_results(public_ev, private_ev, e4_model, REPORT, PUBLIC_REPORT, SOURCE_SHA256,
    experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT, config=config,
    training_summary=training_summary, prepared_fingerprint=prepared.fingerprint,
    training_source=training_source, coverage_metrics=coverage_metrics,
    coverage_params=coverage_params, private_dataset_version=DATASET)
run_id, public_run_id = publication["run_id"], publication["public_run_id"]
if REGISTER_E4:
    registration = register_candidate(REPORT, run_id, public_run_id, experiment=EXPERIMENT)
    print("Candidate version:", registration["version"], "| no alias promoted")
print("E4 complete · private:", run_id, "| public:", public_run_id)

🏃 View run E4 · clip-vit-b32-crop-lora-e3-v4.100@v2 on pettrace-training@v1 at: https://ml.pettrace.app/#/experiments/3/runs/2302b302772b4349894e1d1b58d438ed
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
🏃 View run E4 · clip-vit-b32-crop-lora-e3-v4.100@v2 on pettrace-reid@v4 at: https://ml.pettrace.app/#/experiments/3/runs/5bc216d03afc4a31a15c53fc3bfbc747
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
Drive: backup confirmed (46 files, 01793993541265614828-639bf0fecc1448d1aa3a59eaa90675af.tar).


2026/11/06 19:32:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pettrace-embedder, version 19


Drive: backup confirmed (1 files, 01793993549732190492-8a735c245599492c86dea41bb8ec17da.tar).
Candidate version: 19 | no alias promoted
E4 complete · private: 5bc216d03afc4a31a15c53fc3bfbc747 | public: 2302b302772b4349894e1d1b58d438ed
